# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# ensures you have llama pulled
!ollama pull llama3.2

In [ ]:
MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

ai_models = [
    ("gpt-4o-mini", MODEL_GPT, "gpt-4o-mini"),
    ("llama3.2", MODEL_LLAMA, "ollama"),
]

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openAI = OpenAI()
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

SYSTEM_PROMPT = """
You're an AI coding tutor.
You help people learn programming with clear and consicise guides to learn a programming concept.
"""

In [ ]:
def stream_response(history, user_message, model_label):
    config_from_label = {label: (model_id, backend) for label, model_id, backend in ai_models}
    model_id, backend = config_from_label.get(model_label, ai_models[0][1:])

    client = openAI if backend == "gpt-4o-mini" else ollama
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user, assistant in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": assistant or ""})
    messages.append({"role": "user", "content": user_message})

    stream = client.chat.completions.create(
        model=model_id,
        messages=messages,
        stream=True,
    )
    for chunk in stream:
        part = chunk.choices[0].delta.content or ""
        if part:
            yield part

In [ ]:
def chat(message, history, model_choice):
    if not message or not message.strip():
        return
    full = ""
    for chunk in stream_response(history, message, model_choice):
        full += chunk
        yield full

In [ ]:
model_selector = gr.Dropdown(
    choices=[label for label, _, _ in ai_models],
    value=ai_models[0][0],
    label="Model",
)

ui = gr.ChatInterface(
    chat,
    additional_inputs=[model_selector],
    title="AI Coding Tutor",
    description="What would you like to learn today?",
)
ui.launch(share=True)